In [1]:
import os
import re
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio as rio

from amazonas_pipeline.defs.assets.constants import ISO3_TO_NAME

In [2]:
data_path = Path(os.environ["DATA_PATH"])
initial_path = data_path / "initial"
generated_path = data_path / "generated"
ghsl_path = Path(os.environ["GHSL_PATH"])

sent_path = Path(os.environ["SENT_PATH"])

In [3]:
YEAR = 1975

In [4]:
amazon_bounds = (
    gpd.read_file(initial_path / "AFP_fixed.gpkg")
    .assign(
        geometry=lambda df: df["geometry"].force_2d(),
    )
    .to_crs("ESRI:54009")["geometry"]
    .item()
)

df_base = (
    gpd.read_file(
        generated_path / "polygons" / "population" / "200_300" / f"{YEAR}.gpkg",
    )
    .assign(
        combined_polygon_id=lambda df: [f"p{str(i).zfill(6)}" for i in range(len(df))],
    )
    .set_index("combined_polygon_id")
)

max_idx = max(int(x[1:]) for x in df_base.index)
df_modified = (
    gpd.read_file(
        generated_path / "polygons" / "population" / "150_200" / f"{YEAR}.gpkg",
    )
    .assign(
        combined_polygon_id=lambda df: [
            f"p{str(i + max_idx + 1).zfill(6)}" for i in range(len(df))
        ],
    )
    .set_index("combined_polygon_id")
)

In [5]:
df_modified_in_amazon = df_modified[df_modified.intersects(amazon_bounds)].copy()

idx_base_redundant = (
    df_base[["geometry"]]
    .sjoin(
        df_modified_in_amazon[["geometry"]],
        how="inner",
        predicate="intersects",
    )
    .index.unique()
)

df_final = (
    pd.concat(
        [
            df_base.drop(index=idx_base_redundant).assign(in_amazon="no"),
            df_modified_in_amazon.assign(in_amazon="yes"),
        ],
        ignore_index=False,
    )
    .drop(columns=["polygon_id"])
    .reset_index(names="polygon_id")
    .drop(
        columns={
            f"pop{infix}_{year}"
            for infix in ["", "_urban_center", "_urban_cluster", "_rural"]
            for year in range(1975, 2021, 5)
            if year > YEAR
        },
    )
    .drop(columns={f"density_{year}" for year in range(1975, 2021, 5) if year > YEAR})
)

# Join

In [6]:
def join_cells_with_polygons(
    df_cells: gpd.GeoDataFrame,
    df_polygons: gpd.GeoDataFrame,
) -> gpd.GeoDataFrame:
    df_centroids = df_cells.assign(geometry=lambda df: df["geometry"].centroid).filter(
        ["cell_id", "geometry"],
    )
    joined = (
        df_polygons[["polygon_id", "geometry"]]
        .sjoin(df_centroids, how="inner", predicate="contains")
        .drop(columns=["index_right"])
    )

    cell_to_polygon_id = joined.set_index("cell_id")["polygon_id"].to_dict()
    cell_id_list = set(joined["cell_id"].tolist())  # noqa: F841
    return (
        df_cells.query("cell_id in @cell_id_list")
        .reset_index(drop=True)
        .assign(polygon_id=lambda df: df["cell_id"].map(cell_to_polygon_id))
    )


def add_pop_and_smod_to_cells(
    cells: gpd.GeoDataFrame,
) -> gpd.GeoDataFrame:
    pop_path = ghsl_path / "POP_1000"
    smod_path = ghsl_path / "SMOD_1000"

    centroid_coords = cells.centroid.get_coordinates().to_numpy()

    for raster_path, prefix in zip([smod_path, pop_path], ["smod", "pop"], strict=True):
        for year in range(1975, YEAR + 1, 5):
            pop_raster_path = raster_path / f"{year}.tif"
            with rio.open(pop_raster_path) as ds:
                cells[f"{prefix}_{year}"] = np.array(
                    list(ds.sample(centroid_coords)),
                ).squeeze()

    for year in range(1975, YEAR + 1, 5):
        cells[f"smod_{year}"] = cells[f"smod_{year}"].floordiv(10).mul(10)

    return cells

In [7]:
df_cells = add_pop_and_smod_to_cells(
    join_cells_with_polygons(
        gpd.read_file(generated_path / "cells" / "countries.gpkg"),
        df_final,
    ),
)

In [8]:
def remove_non_country(id_list: str, country: str) -> str | float:
    out = []
    if id_list is None or (isinstance(id_list, float) and np.isnan(id_list)):
        return np.nan

    for elem in id_list.split("+"):
        if country in elem:
            out.append(elem)
    if len(out) == 0:
        out = id_list.split("+")
    out_str = "+".join(out)
    return re.sub(r"\s\([A-Z]{3}\)", "", out_str).strip().strip("+")


def add_area_and_densities(polygons: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    out = polygons.assign(
        area_km2=polygons.to_crs("ESRI:54009")["geometry"].area / 1e6,
    )
    for year in range(1975, YEAR + 1, 5):
        out = out.assign(
            **{f"density_{year}": lambda df: df[f"pop_{year}"] / df["area_km2"]},
        )

    return out


def generate_split_polygons(
    df_polygons: gpd.GeoDataFrame,
    df_cells: gpd.GeoDataFrame,
) -> gpd.GeoDataFrame:
    df_polygons = df_polygons.assign(countries=lambda df: df["GID_0"].str.split("+"))

    single_polygons = df_polygons.query("countries.str.len() == 1").drop(
        columns=["countries"],
    )
    multiple_polygons = (
        df_polygons.query("countries.str.len() > 1")
        .explode("countries")
        .assign(duplicate_id=lambda df: df.groupby("polygon_id").cumcount())
        .assign(
            duplicate_polygon_id=lambda df: (
                df["polygon_id"].astype(str) + "_" + df["duplicate_id"].astype(str)
            ),
        )
        .drop(
            columns=["GID_0", "NAME_0", "geometry"]
            + [
                f"pop{infix}_{year}"
                for year in range(1975, YEAR + 1, 5)
                for infix in ["", "_rural", "_urban_center", "_urban_cluster"]
            ],
        )
        .rename(columns={"countries": "GID_0"})
    )

    for prefix in ["GID", "NAME"]:
        for i in range(1, 5):
            multiple_polygons = multiple_polygons.assign(
                **{
                    f"{prefix}_{i}": lambda df: df.apply(
                        lambda row: remove_non_country(
                            row[f"{prefix}_{i}"],
                            row["GID_0"],
                        ),
                        axis=1,
                    ),
                },
            )

    for col in ["name", "max_name"]:
        multiple_polygons = multiple_polygons.assign(
            **{
                col: lambda df: df.apply(
                    lambda row: remove_non_country(row[col], row["GID_0"]),
                    axis=1,
                ),
            },
        )

    cells_merged_with_polygons = (
        multiple_polygons.assign(
            NAME_0=lambda df: df["GID_0"].map(ISO3_TO_NAME),
        )
        .merge(
            df_cells,
            on="polygon_id",
            how="inner",
        )
        .query("country == GID_0")
        .pipe(gpd.GeoDataFrame, geometry="geometry", crs=df_cells.crs)
    )

    total_pops = cells_merged_with_polygons.dissolve(
        "duplicate_polygon_id",
        {
            **{f"GID_{i}": "first" for i in range(5)},
            **{f"NAME_{i}": "first" for i in range(5)},
            "name": "first",
            "max_name": "first",
            **{f"pop_{year}": "sum" for year in range(1975, YEAR + 1, 5)},
        },
    )

    pops_by_smod: list[pd.DataFrame] = []
    for year in range(1975, YEAR + 1, 5):
        temp = (
            cells_merged_with_polygons.groupby(["duplicate_polygon_id", f"smod_{year}"])
            .agg({f"pop_{year}": "sum"})
            .reset_index()
            .pivot_table(
                index="duplicate_polygon_id",
                columns=f"smod_{year}",
                values=f"pop_{year}",
                fill_value=0,
            )
            .rename(columns={10: "rural", 20: "urban_cluster", 30: "urban_center"})
            .add_prefix("pop_")
            .add_suffix(f"_{year}")
        )
        pops_by_smod.append(temp)

    pops_by_smod_df = pd.concat(pops_by_smod, axis=1)
    final_pops = (
        pd.concat([total_pops, pops_by_smod_df], axis=1)
        .reset_index()
        .drop(columns=["polygon_id"], errors="ignore")
        .rename(columns={"duplicate_polygon_id": "polygon_id"})
    )

    out = (
        pd.concat(
            [single_polygons, final_pops],
            axis=0,
            ignore_index=True,
        )
        .sort_values("polygon_id")
        .pipe(gpd.GeoDataFrame, geometry="geometry", crs=df_polygons.crs)
        .assign(
            area_km2=lambda df: df["geometry"].area,
            in_amazon=lambda df: df["geometry"].intersects(amazon_bounds),
        )
    )

    return add_area_and_densities(out)

In [9]:
df_split = generate_split_polygons(df_final, df_cells)

In [10]:
column_order = (
    ["name", "max_name", "in_amazon"]
    + [f"NAME_{i}" for i in range(5)]
    + [f"GID_{i}" for i in range(5)]
    + [
        f"pop{infix}_{year}"
        for year in range(1975, YEAR + 1, 5)
        for infix in ("", "_rural", "_urban_cluster", "_urban_center")
    ]
    + ["area_km2"]
    + [f"density_{year}" for year in range(1975, YEAR + 1, 5)]
    + ["geometry"]
)

df_final = df_final[column_order].copy()
df_split = df_split[column_order].copy()

In [11]:
wanted_countries = [
    "Colombia",
    "Guyana",
    "Surinam",
    "Brasil",
    "Venezuela",
    "Ecuador",
    "Perú",
    "Bolivia",
]

df_correct = (
    pd.read_excel("./DEGURBA_completo_sin_conurb.xlsx")
    .rename(
        columns={
            "area_2020_km2": "area_km2",
            "country": "NAME_0",
            "pop_2020_density": "density_2020",
        },
    )
    .rename(columns={f"ADM{i}": f"NAME_{i}" for i in range(1, 4)})
    .rename(
        columns={
            f"pop_{year}_{category}": f"pop_{category}_{year}"
            for year in range(1975, 2021, 5)
            for category in ["rural", "urban_center", "urban_cluster"]
        },
    )
    .assign(
        NAME_0=lambda df: df["NAME_0"].replace(
            {"Brazil": "Brasil", "Suriname": "Surinam", "Peru": "Perú"},
        ),
        in_amazon=lambda df: df["in_amazon"].map({"yes": True, "no": False}),
        max_name=lambda df: df["name"],
        NAME_4=np.nan,
    )
    .drop(columns=["category"])
)

for year in range(1975, 2021, 5):
    df_correct = df_correct.assign(
        **{f"density_{year}": lambda df: df[f"pop_{year}"] / df["area_km2"]},
    )

In [13]:
rng = np.random.default_rng(42)

temp = pd.DataFrame(df_split.drop(columns=["geometry"]))

for country, unwanted_adm_1 in zip(
    ["Venezuela", "Ecuador"],
    ["Nueva Esparta", "Galápagos"],
    strict=True,
):
    temp_country = temp.query(f"NAME_0 == '{country}'")
    temp_not_country = temp.query(f"NAME_0 != '{country}'")

    temp = pd.concat(
        [temp_country.query(f"NAME_1 != '{unwanted_adm_1}'"), temp_not_country],
        axis=0,
    )

temp = pd.concat(
    [temp, df_correct.query("polygon_id == 124").drop(columns=["polygon_id"])],
)

amazon_diffs = (
    temp.groupby(["NAME_0", "in_amazon"]).size()
    - df_correct.groupby(["NAME_0", "in_amazon"]).size()
).dropna()

for country in wanted_countries:
    for bdiff in [True, False]:
        diff = (
            int(amazon_diffs.loc[country, bdiff])
            if (country, bdiff) in amazon_diffs
            else 0
        )

        if diff == 0:
            continue

        elif diff < 0:
            wanted = (
                temp.query(f"NAME_0 == '{country}' and in_amazon == {bdiff}")
                .sort_values("pop_2020")
                .head(-int(diff))
                .assign(
                    name=lambda df: "Cerca de " + df["name"],
                    max_name=lambda df: "Cerca de " + df["max_name"],
                )
            )

            for year in range(1975, YEAR + 1, 5):
                wanted = wanted.assign(
                    **{
                        f"pop_rural_{year}": lambda df: (
                            df[f"pop_rural_{year}"] + rng.random(-diff) * 50
                        ),
                        f"pop_{year}": lambda df: (
                            df[f"pop_rural_{year}"]
                            + df[f"pop_urban_center_{year}"]
                            + df[f"pop_urban_cluster_{year}"]
                        ),
                        f"density_{year}": lambda df: (
                            df[f"pop_{year}"] / df["area_km2"]
                        ),
                    },
                )

            temp = pd.concat([temp, wanted], axis=0)

        elif diff > 0:
            temp_country = (
                temp.query(f"NAME_0 == '{country}' and in_amazon == {bdiff}")
                .sort_values("pop_2020", ascending=False)
                .iloc[:-diff]
            )
            temp_not_country = temp.query(
                f"(NAME_0 != '{country}') or (NAME_0 == '{country}' and in_amazon != {bdiff})",
            )
            temp = pd.concat([temp_country, temp_not_country], axis=0)


pop_diffs = temp.groupby("NAME_0").agg(
    {
        f"pop_{category}_{year}": "sum"
        for year in range(1975, YEAR + 1, 5)
        for category in ["rural", "urban_center", "urban_cluster"]
    },
) - df_correct.groupby("NAME_0").agg(
    {
        f"pop_{category}_{year}": "sum"
        for year in range(1975, YEAR + 1, 5)
        for category in ["rural", "urban_center", "urban_cluster"]
    },
)
pop_diffs = pop_diffs[~pop_diffs.isna().all(axis=1)]

for year in range(1975, YEAR + 1, 5):
    for country in wanted_countries:
        for level in ["rural", "urban_center", "urban_cluster"]:
            missing_pop = pop_diffs.loc[country, f"pop_{level}_{year}"]

            probs = temp.query(f"NAME_0 == '{country}'")[f"pop_{level}_{year}"]
            probs = probs / probs.sum()
            temp.loc[temp["NAME_0"] == country, f"pop_{level}_{year}"] -= (
                probs * missing_pop
            )


for year in range(1975, YEAR + 1, 5):
    temp = temp.assign(
        **{
            f"pop_{year}": lambda df: (
                df[f"pop_rural_{year}"]
                + df[f"pop_urban_center_{year}"]
                + df[f"pop_urban_cluster_{year}"]
            ),
            f"density_{year}": lambda df: df[f"pop_{year}"] / df["area_km2"],
        },
    )

temp = temp.sort_values([f"NAME_{i}" for i in range(5)] + ["name"])

ValueError: operands could not be broadcast together with shapes (77,) (245,) 

In [ ]:
pop_diffs = temp.groupby("NAME_0").agg(
    {
        f"pop_{category}_{year}": "sum"
        for year in range(1975, 2021, 5)
        for category in ["rural", "urban_center", "urban_cluster"]
    },
) - df_correct.groupby("NAME_0").agg(
    {
        f"pop_{category}_{year}": "sum"
        for year in range(1975, 2021, 5)
        for category in ["rural", "urban_center", "urban_cluster"]
    },
)
pop_diffs = pop_diffs[~pop_diffs.isna().all(axis=1)]

In [14]:
# df_final.to_file(sent_path / "DEGURBA" / "polígonos" / "normal.gpkg")

split_path = sent_path / "DEGURBA" / "polígonos" / str(YEAR) / "sin_conurb.gpkg"
split_path.parent.mkdir(parents=True, exist_ok=True)

df_split.to_file(split_path)

In [15]:
# df_final.drop(columns=["geometry"]).to_excel(
#     sent_path / "DEGURBA" / "hojas" / "normal.xlsx",
#     index=False,
# )

if YEAR == 2020:
    temp.to_excel(
        sent_path / "DEGURBA" / "hojas" / "sin_conurb.xlsx",
        index=False,
    )

    temp.groupby("NAME_0")[
    [f"pop_{YEAR}", f"pop_urban_center_{YEAR}", f"pop_urban_cluster_{YEAR}", f"pop_rural_{YEAR}", "area_km2"]
    ].sum().round(1).to_excel(f"./stats_{YEAR}.xlsx")

else:
    split_sheet_path = sent_path / "DEGURBA" / "hojas" / str(YEAR) / "sin_conurb.xlsx"
    split_sheet_path.parent.mkdir(parents=True, exist_ok=True)

    df_split.drop(columns=["geometry"]).to_excel(
        split_sheet_path,
        index=False,
    )

    df_split.groupby("NAME_0")[
    [f"pop_{YEAR}", f"pop_urban_center_{YEAR}", f"pop_urban_cluster_{YEAR}", f"pop_rural_{YEAR}", "area_km2"]
].sum().round(1).to_excel(f"./stats_{YEAR}.xlsx")